In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


# Experimentação de Modelos

## O que encontrar neste Notebook?

Você encontrará neste notebook um relatório detalhado das experimentações feitas utilizando feature engineering e tunning de hiperparâmetros. Os resultados foram registrados no MLflow para facilitar o tracking desses experimentos.

# Importando as Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import mlflow

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)

from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)


from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


from src.data.make_dataset import process_data
from experiments.run_experiment import run_experiment

# Importando a Base (raw_data)

In [2]:
df = pd.read_excel('../../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


# Processando a Base

In [3]:
df = process_data()

Lendo base raw de: C:\Users\jayne_m2ck4ov\Documents\POS MLE\FASE 1\mle-f1-tech-challenge\data\raw\Telco_customer_churn.xlsx
Iniciando limpeza dos dados...
Verificando dados duplicados...
Nenhum dado duplicado encontrado.
Verificando dados faltantes...
5174 dados faltantes encontrados.
Convertendo Total Charges para numérico (preenchendo vazios com 0)...
Removendo Colunas desnecessárias...
Verificando se há valores nulos...
Nenhum valor nulo encontrado.
===== INFORMAÇÕES GERAIS =====

Total de linhas: 7043
Total de colunas: 24
===== TIPOS DE DADOS =====

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   City               7043 non-null   object 
 2   Gender             7043 non-null   object 
 3   Senior Citizen     7043 non-null   object 
 4   Partner            7043 non-null   objec

C:\Users\jayne_m2ck4ov\Documents\POS MLE\FASE 1\mle-f1-tech-challenge\src\data\make_dataset.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Total Charges"] = pd.to_numeric(df["Total Charges"].replace(" ", np.nan))


# KPI's de Performance

## KPI Técnico - Recall (Sensibilidade)

O principal KPI técnico escolhido, dado a natureza do nosso problema de negócio, foi o **Recall.** Aqui o objetivo é simples, queremos identificar o seguinte cenário: **de todos os clientes que iam churnar, quantos o modelo conseguiu identificar corretamente.** 

Queremos reduzir ao máximo os casos de Falsos Negativos que representam clientes que churnaram e não agimos, resultando na perda de receita.

Essa métrica é crítica para o negócio e permite responder perguntas como: ***"Quanto churn eu estou deixando passar?***"

Para a experimentação vamos considerar modelos com recall > 0.70.

## KPI's Técnicos Secundários

Outras métricas técnicas serão utilizadas para critério de possíveis desempates entre modelos e também para uma melhor interpretação do desempenho nosso modelo:

**`auc`** - avalia o quão bem o modelo consegue separar os clientes que vão churnar dos que não vão (buscamos algo acima de 0,70);

**`precision`** - avalia quantos clientes que foram marcado como churn de fato churnaram. Essa métrica tem um impacto direta no caso de falsos positivos e estabelece um trade-off com `recall`. Ela representa o custo desnecessário com retenção de clientes que não vão churnar;

**`f1-score`** - é uma métrica de equilibrio entre `precision` e `recall`. Ela consegue balancear custo de retenção e perda de clientes, porém não considera o valor financeiro do cliente (assume custos fp e fn, aproximadamente iguais, o que quase nunca é verdade). Como já possuímos métricas financeiras, essa métrica será para uso secundário. Ela nos auxiliará na identificação do quão eficiente o modelo está conseguindo capturar churn.


## KPI de Negócio (definir)

As métricas técnicas presumem que todos os clientes são iguais do ponto de vista financeiro. E, sabemos que isso não ocorre dentro do nosso cenário real do negócio. Dessa forma, o uso dos KPIs econômicos nos permite priorizar clientes de maior valor, reduzir perda financeira real, balancear a relação custo-benefício e tomar decisões mais assertivas com base no negócio.

Em outras palavras, o modelo não só prevê churn, ele prioriza clientes com maior impacto financeiro e maximiza o retorno da retenção. Abaixo temos alguns KPIs econômicos que podemos utilizar para avaliar nosso modelo do ponto de vista do negócio:

- Receita Protegida (Somatória de Montlhy Charges dos TP);
- Receita Perdida (CLTV dos FN);
- Custo de Retenção (FP * Custo_unitário);
- Valor Liquído ou ROI - retorno real gerado pelo modelo (Receita Protegida - Custo de Retenção);
- CLTV Médio dos TP (qualidade dos clientes capturados);
- Taxa de Captura de Valor: quanto do valor em risco conseguimos capturar? (Receita Protegida / (Receita Protegida + Receita Perdida));
- ROI da Retenção ((receita protegida - custo)/custo)

# Metodologia da Experimentação

A experimentação seguiu as seguintes etapas:

1. Treinamento de um modelo de Regressão Logística e Dummy a partir da versão base (sem feature engineering) do dataset para baseline inicial;
2. Versionamento do dataset para cada experimento com tracking das métricas no MLFlow, utilizando a regressão logística como proxy de experimentação;
3. Comparação entre Modelos (regressão logística, ensembles e MLP) a partir da versão final do dataset (best features);
4. Tunning direcionado durante uma hora para o modelo campeão.

## Load config base (base_exp.yaml)

## Inicializando o Experimento MLFlow

In [4]:
mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])
mlflow.set_experiment(config["mlflow"]["experiment_name"])

NameError: name 'config' is not defined

## Baseline

## Feature Engineering

### exp02_add_feat_churn_score

### exp04_add_feat_engagement_score

### exp05_add_tenure_group_encoding_drop_tenure_months

### exp06_add_tranformation_tenure_log_with_tenure_group_enc

### exp07_add_contract_ordinal_encoding

### exp08_add_flag_family_stability

### exp09_add_flag_fiber_no_support

### exp10_add_feat_city_with_mapping_enc

### exp11_add_city_freq_enc_strategy

## MLP (best_features)

# Tunning Hiperparametros